# 02 — Tabular models (baselines, LightGBM)

Runs on the local machine (about 20 s per combination, 65 combinations). Validation only; the test split is not read. Results are organised per model in `results/tabular/` by the last cell.

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "uc04_v2.json").exists())
PY = sys.executable  # run this notebook with the repo venv (.venv)


def run(script, *args):
    """Run a repo script and stream its output; stop on failure."""
    cmd = [PY, str(REPO / "scripts" / script), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", env={**__import__("os").environ, "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed ({p.returncode})")

## 1. Train 65 combinations
`map_threshold` (5), `map_logistic`, `lgbm_numeric`, `lgbm_wave` (4 windows × 5 horizons each). Finished combinations are skipped.

In [ ]:
run("train_tabular.py", "--no-push")

## 2. Both alarm policies and `alarm_time_fraction` (no retraining)

In [ ]:
run("reevaluate_tabular.py")

## 3. Combined table, W*, SHAP, ablation, confidence check

In [ ]:
run("tabular_analysis.py", "--no-push")

## 4. Main comparison with paired bootstrap
wave − numeric, wave − logistic, numeric − threshold.

In [ ]:
run("compare_models.py")

## 5. Exploratory: longer windows (W = 300, 600)
Decided after seeing validation: reported separately, does not change W*.

In [ ]:
run("explore_long_windows.py", "--workers", 6)

## 6. Export shareable results per model

In [ ]:
run("export_results.py")
import pandas as pd
t = pd.read_csv(REPO / "results" / "tabular" / "summary" / "tabular_validation.csv")
t[(t.window == 120) & t.horizon.isin([300, 600])][["model", "horizon", "event_sensitivity", "false_alarms_per_hour", "alarm_ppv", "auroc", "auprc", "alarm_time_fraction"]].round(3)